# Publish training and inference — v2

Enter **bucket, role ARN and image**, then run all six cells in order with a fresh kernel.
This notebook publishes `source-dev-training` and `source-dev-inference`. Start them from Studio afterward.

Use this notebook inside the full application checkout, with the complete updated `mlops-models` and `mlops-features` directories. Processing writes to `/tmp` and uploads its declared
outputs to S3. Batch inference defaults to **Processing**; ordinary Training keeps its separate output handling.

Direct deploy replaces source under the same development name. Rerun between executions, while neither pipeline is running.
For a Feature Store permission workaround, run only the first two code cells of **04A** to
generate the flat sample, then set `data_source="csv_direct"` and `csv_input` below. No ingestion is required.
Both modes produce the same raw snapshot for the remaining training steps.

This edition passes raw/prepared directories and `selected_features.csv` between steps.
Run all cells after updating the source: Training now also mounts selected features.


### 1. Inputs

- Fill in the bucket, exact IAM role ARN and existing ECR image address.
- Leave the optional defaults for the current `source-dev` pipelines and both example models.
- Set a separate job role or VPC/KMS values only if your platform provides them; this cell creates nothing.

In [ ]:
# Platform inputs.
bucket = ""                        # Bucket name, with or without s3://.
role_arn = ""                      # Exact IAM role ARN, including its path.
image_uri = ""                     # Repository address, repository:tag or repository@sha256:...

# Choose Feature Store, or use the flat sample generated in 04A code cell 2.
# csv_direct is explicit: a Feature Store error never silently changes the source.
namespace = "source-dev"
deploy_name = "my-review"
model_ids = None                   # Discover all model folders; or set ["usecase1"].
data_source = "feature_store"     # "feature_store" or "csv_direct".
feature_store_config_uri = ""      # Required only for feature_store: URI printed by 04A.
csv_input = ""                     # For csv_direct: local flat CSV path (04A Local sample) or s3://...csv.
universe_data_root = ""            # Optional s3://bucket/prefix/<ModelId>/universe.csv population root.
job_role_arn = ""                  # Empty uses role_arn for jobs as well as pipelines.
batch_runtime = "processing"      # Use "transform" only after validating native Batch Transform.
infrastructure = {
    "network": {"isolation": False, "subnet_ids": [], "security_group_ids": []},
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
assert data_source in {"feature_store", "csv_direct"}
if data_source == "feature_store":
    assert feature_store_config_uri, "Paste the Feature Store configuration URI printed by 04A."
else:
    assert csv_input, "Supply the Local sample CSV from 04A code cell 2, or its S3 URI."


### 2. Load the application and show the paths

- Find the checkout containing this notebook and import its source publication functions.
- Discover the selected model folders and plan the development source and data locations.
- Print the region, roles and paths; no AWS resources or S3 objects are created.

In [ ]:
from pathlib import Path
import json
import sys
import boto3

bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
assert bucket and "/" not in bucket and role_arn and image_uri, "Fill in bucket, role_arn and image_uri."

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src")]
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, build_definitions,
)
from mlops_skeleton.names import discover_models

model_ids = model_ids or [folder.name for folder in discover_models(ROOT / "mlops-models/src/mlops_models/models")]
pipeline_role_arn = role_arn
job_role_arn = job_role_arn.strip() or role_arn
session = boto3.Session()
sm = session.client("sagemaker")
s3_root = f"s3://{bucket}/{namespace}"
identity = source_identity("direct-deploy", deploy_name)
source_uri = source_location(s3_root, identity)
csv_input_uri, csv_local_path = "", None
if data_source == "csv_direct":
    if csv_input.startswith("s3://"):
        csv_input_uri = csv_input
    else:
        csv_local_path = Path(csv_input).expanduser().resolve()
        assert csv_local_path.is_file(), f"Missing flat sample CSV: {csv_local_path}"
        csv_input_uri = f"{s3_root}/inputs/csv-direct/{deploy_name}/data.csv"

print("Checkout:", ROOT)
print("Region:", session.region_name)
print("Pipeline role:", pipeline_role_arn)
print("Job role:", job_role_arn)
print("Models:", model_ids)
print("Source upload destination:", source_uri)
print("Data source:", data_source)
print("Input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)
print("Population/target root:", universe_data_root or "none; training requires a target column in the data")
print("Run outputs:", s3_root + "/models/<ModelId>/runs/<execution-id>/")
print("Source identity:", identity)

### 3. Select the existing image

- Keep a digest URI as supplied, or resolve a tag to its exact digest.
- For a repository address alone, select its newest tagged image and print the selection.
- Use that image in the definitions; no image is built or modified.

In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image, as 00G section 6 does.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)

### 4. Build both pipeline definitions

- Generate training as GetData → FeatureSelection → Preprocess → Train/Tune → Evaluate → Registry.
- Generate the approved-model batch inference pipeline from the same checkout.
- Pass raw/prepared directories and selected features; upload every file in each declared output, including reports.
- Save the two JSON definitions locally and print their paths; this does not publish them yet.

In [ ]:
definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
    feature_store_config_uri=feature_store_config_uri, universe_data_root=universe_data_root,
    data_source=data_source, csv_input_uri=csv_input_uri,
)
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Registry groups:", [namespace + "-" + model for model in model_ids])

print("Batch inference runtime:", batch_runtime)
print("UseHPO defaults to false; validate the image's tuning startup before enabling it.")

## 5. Upload the source and optional flat CSV

- Upload the complete Models package to the displayed source prefix.
- Include each model's configuration and the job's source loader.
- In csv_direct mode, upload the local flat CSV to the printed input URI (or reuse the supplied S3 object).
- GetData reads its selected source when the pipeline runs; later steps keep their existing handoffs.

In [ ]:
uploaded_source = upload_source(session, s3_root, identity)
print("Uploaded source:", uploaded_source)
if csv_local_path is not None:
    csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
    kms = infrastructure.get("output_kms_key_arn", "")
    encryption = {"ServerSideEncryption": "aws:kms", "SSEKMSKeyId": kms} if kms else {}
    session.client("s3").upload_file(str(csv_local_path), csv_bucket, csv_key, ExtraArgs=encryption)
    print("Uploaded flat CSV:", csv_input_uri)
print("GetData / batch Preprocess input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)

### 6. Publish both pipelines

- Create the pipelines, or update existing pipelines owned by this namespace and deployment mode.
- Use ordinary CreatePipeline/UpdatePipeline calls; version-label permissions are not required.
- Read back the definitions and roles, save the publication record, and print the Studio pipeline names.

In [ ]:
published = {}
for kind, body in definitions.items():
    name = f"{namespace}-{kind}"
    request = {
        "PipelineName": name, "RoleArn": pipeline_role_arn,
        "PipelineDefinition": json.dumps(body),
    }
    try:
        existing = sm.describe_pipeline(PipelineName=name)
    except sm.exceptions.ClientError as error:
        if "does not exist" not in error.response["Error"]["Message"]:
            raise
        response = sm.create_pipeline(**request, Tags=[
            {"Key": "Deployment", "Value": namespace},
            {"Key": "SourceDeploy", "Value": identity["mode"]},
        ])
        version = 1  # The initial version of a newly created pipeline.
    else:
        tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=existing["PipelineArn"])["Tags"]}
        if tags.get("Deployment") != namespace or tags.get("SourceDeploy") != identity["mode"]:
            raise ValueError(f"{name} belongs to another deployment.")
        response = sm.update_pipeline(**request)
        version = response["PipelineVersionId"]
    actual = sm.describe_pipeline(PipelineName=name, PipelineVersionId=version)
    assert json.loads(actual["PipelineDefinition"]) == body, "Published definition differs; inspect before executing."
    assert actual["RoleArn"] == pipeline_role_arn, "Saved pipeline role differs; inspect before executing."
    published[kind] = {"name": name, "arn": response["PipelineArn"], "version": version}
    print("Published:", name, "| version:", version, "| saved role:", actual["RoleArn"])

publication = {
    "namespace": namespace, "pipelines": published, "models": model_ids,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
    "feature_store_config_uri": feature_store_config_uri, "universe_data_root": universe_data_root,
    "data_source": data_source, "csv_input_uri": csv_input_uri,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication complete:", path.resolve())
print("Both pipelines are ready to start from Studio.")

### Continue in Studio

1. Open **Pipelines → source-dev-training → Create execution**. Select `ModelId`, keep `UseHPO=false`,
   and name the execution `<ModelId>-training-<label>`. If you changed the namespace, use the name printed above.
2. In GetData's logs, look for **`processing_outputs_uploaded`**. This confirms that its output upload finished.
   Inspect `data/raw/data.csv` and `provenance.json` under this execution in either input mode.
3. Review the evaluation and pending model under **Model Registry → `<namespace>-<ModelId>`**, then approve that version.
4. Start **`<namespace>-inference`** with the same `ModelId`. It selects the latest Completed, Approved model;
   if none exists, it fails before scoring.

In Feature Store mode, leave **FeatureStoreDeliveryId** blank for the latest completed ingestion,
or supply a delivery ID to reproduce a specific snapshot. **UniverseDataRoot** may differ between
training and inference. Subsequent runs query the store without running 04A again.

In **csv_direct** mode, **CsvInputUri** points to the flat input CSV. SageMaker downloads it into
GetData's input folder. GetData selects the model columns and saves the same raw CSV/output paths;
FeatureSelection, Preprocess and Train stay unchanged. Training needs a target column in that CSV.
For supplied schemas, set the model's configured keys/features/label to match the generated headers.
Inference uses the approved model's saved configuration and source, so train and approve a new version first.
Changing between feature_store and csv_direct requires rerunning this publication notebook.

Publication does not run jobs or approve models. Use **operations/03G** only if you want notebook-based execution and inspection.